# A4 — Policy learning y uplift: política óptima bajo presupuesto (IHDP)

**Curso:** Sistema de Inteligencia Causal · *profesionales*  
**Sesiones:** S22–S23 · **Módulo 4**  
**Lectura monografía:** Cap. **21** Policy learning · Cap. **22** Uplift · Cap. **25** Decisiones (contexto)  
**Labs:** L21 · L22 · L23  
**Dataset:** `real/ihdp` (continúa A3)  
**Técnica:** top-k / umbral bajo presupuesto, welfare, Qini, baselines, equidad  
**Alimenta Serie B:** **B3** (`artifacts/A4/policy.joblib` + servicio local)

### Pregunta de valor (disrupción)

> *Con capacidad para tratar solo el 20%, ¿a quién asignamos el cupo?*  
> *Compara política causal vs aleatoria vs “tratar al de peor outcome predicho”.*

### Objetivos de aprendizaje

1. Convertir CATE en política $\pi(x)$ bajo presupuesto $B$.
2. Evaluar welfare (IPW sin truth; oracle $\tau$ solo al final).
3. Comparar baselines **aleatorio** y **outcome-only** (no causal).
4. Auditar equidad simple de la política.
5. Serializar política + *policy card* para B3.


### Decisión de negocio

> *Si solo predijéramos el outcome, el presupuesto se asignaría a quienes tienen peor pronóstico ("outcome-only") — el error clásico de "tratar al más enfermo".*  
> *Con el efecto (política top-k sobre CATE), la decisión de negocio es distinta: el 20% de capacidad se asigna a quien más se beneficia de la intervención, medido en welfare, no en necesidad aparente.*

Esta comparación (§5 del notebook: `cate_topk` vs `random` vs `outcome_only`) es la evidencia cuantitativa que sostiene la decisión frente a negocio.

## Aviso sobre los datos (transparencia)

Este notebook usa un escenario del **Causal Benchmark Ecosystem**.

Los archivos en `datasets/real/` **no son datos clínicos ni administrativos reales**.  
Lo auténtico de la literatura son **nombres de variables estilo Hill (IHDP), diseño del estudio y órdenes de magnitud de referencia**.  
Los valores numéricos se generan con SEM / fórmulas documentadas para que exista **ground truth** (`truth_*.csv`) usable en evaluación y en el evaluación docente.

| Capa | Archivos | Quién la ve en este notebook |
|------|----------|------------------------------|
| Observables | `observed_*.csv` | **Siempre** (flujo del estudiante) |
| Ground truth | `truth_*.csv` (`y0`, `y1`, `tau`) | **Solo** sección final marcada «evaluación con ground truth / welfare oracle» |
| Identificación / scorecard | `identification.json`, `metadata.json`, `scorecard.json` | Lectura guiada |
| Artefactos A3 | `artifacts/A3/cate_scores_test.csv` | Ranking de política (preferido) |

Consulta `metadata.json` → campos `what_is_real` / `what_is_synthetic`.

**Política de truth:** no uses `y0` / `y1` / `tau` fuera de la sección final de evaluación.


## 1. Setup y carga del escenario

Semilla global del curso: `RANDOM_STATE = 42`.  
Splits oficiales del escenario; no re-particionamos.  
Presupuesto por defecto del curso: \(B = 0.20\) (tratar al top 20% por \(\hat\tau\)).

Si existen scores de A3 se reutilizan; si no, se re-estima un T-learner embebido (mismo contrato del curso).


In [ ]:
# Paths relativos al monorepo (kernel-less: funciona con nbconvert/pytest)
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_NOTEBOOK_DIR = Path.cwd()
_CANDIDATES = [
    _NOTEBOOK_DIR.parent,  # .../03_notebooks_caso_estudio if cwd is notebooks/
    _NOTEBOOK_DIR,         # if cwd is already package root
]
for _p in _CANDIDATES:
    if _p is None:
        continue
    if (_p / "lib" / "paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        PACKAGE_ROOT = _p
        break
else:
    here = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for parent in [here, *here.parents]:
        if (parent / "lib" / "paths.py").is_file():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            PACKAGE_ROOT = parent
            break
    if PACKAGE_ROOT is None:
        raise RuntimeError("Could not locate package root (lib/paths.py)")

from lib.paths import (
    RANDOM_STATE,
    SCENARIO_A4,
    artifacts_dir,
    ensure_dir,
    package_root,
)
from lib.data_io import (
    assert_no_truth_columns,
    load_scenario,
)
from lib.metrics import qini_auc, qini_curve, uplift_curve
from lib.plotting import apply_course_style, plot_qini, plot_uplift_bins
from lib.policy_utils import (
    apply_policy,
    compare_policies,
    equity_table,
    export_a4_artifacts,
    load_or_compute_cate_scores,
    load_policy,
    make_threshold_policy,
    outcome_only_policy,
    policy_value_ipw,
    random_policy,
    save_policy,
    build_rich_policy_card,
    threshold_from_budget,
)

assert RANDOM_STATE == 42
SCENARIO = SCENARIO_A4  # real/ihdp
BUDGET_B = 0.20
METHOD_SELECTED = "t_learner"  # contrato del curso (A3 → A4)
ART_DIR = ensure_dir(artifacts_dir("A4"))
A3_SCORES = artifacts_dir("A3") / "cate_scores_test.csv"

print("PACKAGE_ROOT :", PACKAGE_ROOT)
print("SCENARIO     :", SCENARIO)
print("RANDOM_STATE :", RANDOM_STATE)
print("BUDGET_B     :", BUDGET_B)
print("ART_DIR      :", ART_DIR)
print("A3 scores    :", A3_SCORES, "exists=", A3_SCORES.is_file())


In [ ]:
# Carga SOLO observables + sidecars (sin truth)
bundle = load_scenario(SCENARIO)
assert_no_truth_columns(bundle.train, context="train")
assert_no_truth_columns(bundle.validation, context="validation")
assert_no_truth_columns(bundle.test, context="test")

meta = bundle.meta
ident = bundle.identification
scorecard = bundle.scorecard

print("scenario_id :", bundle.scenario_id)
print("shapes      : train", bundle.train.shape,
      "| val", bundle.validation.shape,
      "| test", bundle.test.shape)
print("description :", meta.get("description") or meta.get("full_name"))
print("what_is_real     :", meta.get("what_is_real"))
print("what_is_synthetic:", meta.get("what_is_synthetic"))
print("ate_true (metadata, no usar en estimación):", meta.get("ate_true"))


In [ ]:
# Roles de columnas y matrices de trabajo
Y_COL = bundle.outcome_column()    # outcome
T_COL = bundle.treatment_column()  # treatment
X_COLS = bundle.covariate_columns()

print(f"Y = {Y_COL!r}  |  T = {T_COL!r}")
print(f"X ({len(X_COLS)}): {X_COLS}")

Y_tr, T_tr, X_tr = bundle.ytx("train")
Y_va, T_va, X_va = bundle.ytx("validation")
Y_te, T_te, X_te = bundle.ytx("test")

print("train treatment rate:", float(T_tr.mean()))
print("test n              :", len(Y_te))
assert list(X_tr.columns) == X_COLS


## 2. Cargar scores CATE de A3 (o reestimar)

Preferimos `artifacts/A3/cate_scores_test.csv` con `method_selected = t_learner`.  
Si el archivo no existe, `load_or_compute_cate_scores` ajusta un T-learner en train y scorea test.
También devuelve un **scorer picklable** para `policy.predict(X=...)` en B3.


In [ ]:
scores_df, tau_hat, method_name, scorer = load_or_compute_cate_scores(
    SCENARIO,
    a3_scores_path=A3_SCORES if A3_SCORES.is_file() else None,
    method_selected=METHOD_SELECTED,
    random_state=RANDOM_STATE,
)

print("method_name :", method_name)
print("n scores    :", len(tau_hat))
print("mean tau_hat:", float(np.mean(tau_hat)))
print("std  tau_hat:", float(np.std(tau_hat)))
print("scores_df columns:", list(scores_df.columns)[:8], "...")
assert len(tau_hat) == len(X_te), "CATE scores must align with test rows"
assert np.isfinite(tau_hat).all()


## 3. De CATE a política bajo presupuesto \(B\)

Regla miópica (top-k / umbral calibrado):

\[
\lambda = \mathrm{Quantile}_{1-B}\big(\hat\tau(x)\big), \quad
\pi(x) = \mathbf{1}\{\hat\tau(x) \ge \lambda\}.
\]

Con \(B=0.20\) tratamos aproximadamente el 20% con mayor uplift estimado.
También puedes explorar \(B \in \{0.10, 0.20, 0.30\}\).


In [ ]:
# Política óptima miópica bajo B
policy = make_threshold_policy(
    tau_hat,
    budget_fraction=BUDGET_B,
    mode="topk",
    method_name=method_name,
    feature_names=X_COLS,
    score_model=scorer,
    meta={
        "scenario": SCENARIO,
        "scenario_id": bundle.scenario_id,
        "budget_fraction": BUDGET_B,
        "method_selected": method_name,
        "random_state": RANDOM_STATE,
        "split": "test",
    },
)

pred = policy.predict(scores=tau_hat)
decision = pred["decision"]
treat_rate = float(np.mean(decision))

print(f"mode           : {policy.mode}")
print(f"threshold λ    : {policy.threshold:.6f}")
print(f"budget B       : {policy.budget_fraction}")
print(f"treat_rate π   : {treat_rate:.4f}  (target {BUDGET_B} ± 0.02)")
print(f"n_treated      : {int(decision.sum())} / {len(decision)}")

# Sensibilidad a B
rows_b = []
for b in (0.10, 0.20, 0.30):
    thr = threshold_from_budget(tau_hat, b)
    d = (tau_hat >= thr).astype(int)
    rows_b.append({"B": b, "threshold": thr, "treat_rate": float(d.mean()), "n_treated": int(d.sum())})
print(pd.DataFrame(rows_b).to_string(index=False))


## 4. Curvas de uplift / Qini (sin truth)

Validación offline del **ranking** por \(\hat\tau\): curva Qini y uplift por bins sobre outcomes factuales del test.
Esto **no** demuestra identificación de \(\tau(x)\); solo que el score ordena incrementalmente mejor que el azar bajo los datos observados.


In [ ]:
apply_course_style()

curve = qini_curve(Y_te, T_te, tau_hat)
q_auc = qini_auc(curve)
uplift_df = uplift_curve(Y_te, T_te, tau_hat, n_bins=10)

print(f"Qini AUC (vs random baseline area): {q_auc:.4f}")
print(uplift_df.head().to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))
plot_qini(curve, ax=axes[0], title=f"Qini (test) — {method_name}")
plot_uplift_bins(uplift_df, ax=axes[1], title="Uplift by score bin (test)")
fig.suptitle(f"A4 ranking diagnostics | B={BUDGET_B:.0%}", fontsize=11)
fig.tight_layout()
uplift_png = ART_DIR / "uplift_curve.png"
fig.savefig(uplift_png, dpi=120)
print("wrote", uplift_png)
plt.show()


## 5. Baselines: random y outcome-only

| Política | Idea |
|----------|------|
| **cate_topk** | Tratar top-\(B\) por \(\hat\tau(x)\) (causal) |
| **random** | Tratar \(B\) al azar |
| **outcome_only_low_y** | Tratar a quienes tienen **peor** outcome predicho (necesidad / riesgo no causal) |

El baseline outcome-only es el error clásico de *tratar al más enfermo* sin modelar el **efecto** del tratamiento.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

# Risk model: E[Y|X] on train → predict test (no treatment)
risk_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
risk_model.fit(np.asarray(X_tr, dtype=float), np.asarray(Y_tr, dtype=float))
y_risk = risk_model.predict(np.asarray(X_te, dtype=float))

dec_random = random_policy(len(tau_hat), BUDGET_B, random_state=RANDOM_STATE)
dec_risk_low = outcome_only_policy(y_risk, BUDGET_B, treat_highest_risk=False)
dec_risk_high = outcome_only_policy(y_risk, BUDGET_B, treat_highest_risk=True)

# Propensity for IPW value (teaching estimator; observed only)
prop_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
prop_clf.fit(np.asarray(X_tr, dtype=float), np.asarray(T_tr, dtype=int))
e_te = prop_clf.predict_proba(np.asarray(X_te, dtype=float))[:, 1]

policies = {
    "cate_topk": decision,
    "random": dec_random,
    "outcome_only_low_y": dec_risk_low,
    "outcome_only_high_y": dec_risk_high,
}

cmp_ipw = compare_policies(
    policies,
    y=Y_te,
    t=T_te,
    propensity=e_te,
)
print("--- Policy comparison (IPW value; no truth) ---")
print(cmp_ipw.to_string(index=False))
print(
    "\nInterpretación: value_ipw es un estimador *teaching* de alineación "
    "política–tratamiento; el ranking Qini y el welfare oracle (sección final) "
    "completan la evaluación."
)


## 6. Análisis de equidad (descriptivo)

Las covariables del escenario están en escala estandarizada del SEM.  
Auditamos **tasa de tratamiento bajo \(\pi\)** por:

- terciles de `bw` (birth weight proxy),
- split por mediana de `sex`,
- split por mediana de `b.marr`.

Esto **no** certifica fairness legal ni ausencia de disparidad injusta; es un control de sanidad para discutir concentración de beneficios.


In [ ]:
eq_frame = X_te.copy()
eq_frame["bw_tercile"] = pd.qcut(
    eq_frame["bw"],
    q=3,
    labels=["T1_low", "T2_mid", "T3_high"],
    duplicates="drop",
)
sex_med = float(np.median(eq_frame["sex"]))
eq_frame["sex_bin"] = np.where(eq_frame["sex"] >= sex_med, "sex_high", "sex_low")
bm_med = float(np.median(eq_frame["b.marr"]))
eq_frame["bmarr_bin"] = np.where(eq_frame["b.marr"] >= bm_med, "b.marr_high", "b.marr_low")

eq_bw = equity_table(decision, eq_frame["bw_tercile"], group_name="bw_tercile")
eq_sex = equity_table(decision, eq_frame["sex_bin"], group_name="sex_bin")
eq_bm = equity_table(decision, eq_frame["bmarr_bin"], group_name="bmarr_bin")

print("--- treat rate by bw tercile ---")
print(eq_bw.to_string(index=False))
print("--- treat rate by sex median split ---")
print(eq_sex.to_string(index=False))
print("--- treat rate by b.marr median split ---")
print(eq_bm.to_string(index=False))

# Tabla exportable unificada
parts = []
for gname, tbl in (
    ("bw_tercile", eq_bw),
    ("sex_median_split", eq_sex.rename(columns={"sex_bin": "group"}) if "sex_bin" in eq_sex.columns else eq_sex),
    ("b.marr_median_split", eq_bm.rename(columns={"bmarr_bin": "group"}) if "bmarr_bin" in eq_bm.columns else eq_bm),
):
    p = tbl.copy()
    # normalize
    for c in list(p.columns):
        if c not in ("n", "n_treated", "treat_rate", "group"):
            p = p.rename(columns={c: "group"})
    if "group" not in p.columns:
        p = p.rename(columns={p.columns[0]: "group"})
    p.insert(0, "group_var", gname)
    parts.append(p[["group_var", "group", "n", "n_treated", "treat_rate"]])

equity_out = pd.concat(parts, ignore_index=True)
equity_path = ART_DIR / "equity_table.csv"
equity_out.to_csv(equity_path, index=False)
print("wrote", equity_path)
print(equity_out.to_string(index=False))


## 7. Export: `policy.joblib` + policy card (artefacto B3)

Interfaz contractual para Serie B:

```python
out = policy.predict(X=X_new)  # o policy.predict(scores=s)
# out["decision"], out["uplift_score"]
```

El policy card documenta alcance, features, umbral, supuestos, riesgos y **no-claims**.


In [ ]:
# Serializar política + card rica (también regenerable con export_a4_artifacts)
card_md = build_rich_policy_card(
    scenario_id=bundle.scenario_id,
    scenario=SCENARIO,
    budget_fraction=BUDGET_B,
    method_name=method_name,
    threshold=float(policy.threshold),
    mode=str(policy.mode),
    feature_names=X_COLS,
    treat_rate_test=treat_rate,
    n_test=int(len(tau_hat)),
    equity_group="bw_tercile",
    random_state=RANDOM_STATE,
)

policy_path = ART_DIR / "policy.joblib"
save_policy(policy, policy_path, policy_card=card_md)
print("wrote", policy_path)
print("wrote", ART_DIR / "policy_card.md")

# Export de referencia (idempotente: reescribe joblib, card, plot, equity)
export_info = export_a4_artifacts(
    scenario=SCENARIO,
    out_dir=ART_DIR,
    budget_fraction=BUDGET_B,
    method_selected=METHOD_SELECTED,
    a3_scores_path=A3_SCORES if A3_SCORES.is_file() else None,
    random_state=RANDOM_STATE,
    save_plot=True,
)
print("export_a4_artifacts:", {k: export_info[k] for k in (
    "policy_path", "policy_card_path", "uplift_curve_path", "equity_table_path",
    "method_selected", "threshold", "treat_rate_test", "qini_auc_test", "a3_scores_used",
)})

# Smoke B3: cargar y predecir 3 filas
pol_loaded = load_policy(policy_path)
smoke = pol_loaded.predict(X=X_te.iloc[:3])
print("smoke decision:", smoke["decision"].tolist())
print("smoke scores  :", [round(float(s), 4) for s in smoke["uplift_score"]])


## 8. Discusión guiada (escribir en el entregable)

Completa en ½–1 página:

1. **Presupuesto:** si \(B\) baja de 0.20 a 0.10, ¿quién deja de ser tratado y por qué importa para welfare?
2. **Baseline outcome-only:** ¿en qué situación clínica/económica tratar al de *peor* outcome predicho es engañoso?
3. **Equidad:** ¿la política concentra tratamientos en algún tercil de `bw`? ¿Es eso indeseable *a priori* o reflejo de heterogeneidad real del efecto?
4. **Handoff B3:** ¿qué información del policy card no debería omitirse al servir una policy local?

### Respuesta de referencia (modelo — adaptar con tus números)

> Con menor \(B\) solo entran unidades con \(\hat\tau\) más alto; el welfare oracle por unidad tratada suele subir pero el impacto poblacional total puede caer.  
> Outcome-only ignora que el efecto del programa puede ser mayor en perfiles con outcome basal aceptable.  
> Diferencias de treat-rate por tercil de `bw` son una señal para auditoría, no una prueba de sesgo.  
> El card debe declarar datos sintéticos, \(B\), features, supuestos y non-claims.


## Sección de evaluación reservada

La edición pública termina antes de las celdas que requieren `truth`, `y0`, `y1` o `tau` simulados. La evaluación docente se ejecuta en un entorno separado. Continúa la interpretación con datos observados, incertidumbre y supuestos declarados.
